# House AI — HunyuanWorld Colab backend\nUse a GPU runtime. This notebook installs HunyuanWorld, starts the House AI bridge, and exposes it through a temporary Cloudflare tunnel. Copy the printed URL into House AI.\n

In [ ]:
!nvidia-smi\nimport torch\nassert torch.cuda.is_available(), 'A CUDA GPU runtime is required. In Colab choose Runtime > Change runtime type > GPU.'\nprint('GPU:', torch.cuda.get_device_name(0))\nprint('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))\n

In [ ]:
!git clone --depth 1 https://github.com/Tencent-Hunyuan/HunyuanWorld-1.0.git /content/HunyuanWorld-1.0\n%cd /content/HunyuanWorld-1.0\n!pip install -r requirements.txt\n

In [ ]:
# Download model weights using the project's official helper.\n!python3 scripts/download_models.py\n

In [ ]:
!git clone --depth 1 https://github.com/Rohit9605/RohitGundam-house-ai.git /content/house-ai\nimport os\nos.environ['HUNYUAN_WORLD_DIR']='/content/HunyuanWorld-1.0'\nget_ipython().system('python /content/house-ai/local-world/server.py > /content/house-ai-server.log 2>&1 &')\n

In [ ]:
# Expose port 8787. Keep this cell/runtime alive while using House AI.\n!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared\n!chmod +x /usr/local/bin/cloudflared\nimport subprocess, re, time\np=subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:8787','--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)\nurl=None\nfor _ in range(60):\n    line=p.stdout.readline(); print(line,end='')\n    m=re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com',line)\n    if m: url=m.group(0); break\nassert url, 'Tunnel URL was not created.'\nprint('\\nCOPY THIS INTO HOUSE AI:\\n',url)\n